# Guided TVM Fixed-y Replication

This notebook uses the same `qmm.qmm_json_helper` builder/runner flow as `qmm_guided_json_runner.ipynb`, but it is preconfigured to reproduce the four-panel TVM fixed-`y` figure style:

1. build the branch-specific JSON configs from one guided-style request
2. run or reuse the `equal_b` and `target_l` asymmetric quarkyonic outputs
3. plot `epsilon / rho_B - m_N` and `v_s^2` for several fixed proton fractions
4. add insets showing the peak `v_s^2` versus the density of that peak as `y` varies

No QMM library code is changed here. The notebook only drives the existing workflow and postprocesses its CSV outputs.

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

candidate_roots = []
seen = set()
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (base, base / 'QMM'):
        resolved = candidate.resolve()
        marker = str(resolved)
        if marker not in seen:
            seen.add(marker)
            candidate_roots.append(resolved)

ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'qmm').exists() and (candidate / 'examples').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError('Could not find the QMM project root from this notebook.')

SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qmm.constants import DEFAULT_PHYSICAL_CONSTANTS
from qmm.qmm_json_helper import build_configs, default_request, run_configs, save_configs, summarize_run

PYTHON_BIN = sys.executable
NOTEBOOK_OUTPUT_DIR = ROOT / 'results' / 'generated' / 'guided_tvm_fixed_y_replication'
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'QMM root: {ROOT}')
print(f'Notebook output directory: {NOTEBOOK_OUTPUT_DIR.relative_to(ROOT)}')
print(f'Runner Python: {PYTHON_BIN}')

In [ ]:
MAIN_Y_VALUES = [0.5, 0.4, 0.3, 0.2, 0.1, 0.0]
DENSE_Y_VALUES = [round(float(value), 3) for value in np.linspace(0.0, 0.5, 21)]
BASE_RUN_NAME = 'guided_tvm_fixed_y_replication'
BASE_OUTPUT_SUBDIR = 'guided_tvm_fixed_y_replication'

guided_request = default_request()
guided_request.update(
    {
        'run_name': BASE_RUN_NAME,
        'output_subdir': BASE_OUTPUT_SUBDIR,
        'mean_field': 'vdw',
        'excluded_volume': 'tvm',
        'model_name': 'tvm',
        'preset': 'asymmetric_fixed_y',
        'branch_mode': 'both',
        'parameter_mode': 'none',
        'parameter_value': None,
        'target_k0': None,
        'quick_mode': False,
        'momentum_mode': 'quarkyonic',
        'lambda_momentum_mev': 200.0,
        'beta_equilibrium': False,
        'proton_fraction_values': DENSE_Y_VALUES,
        'write_json': True,
        'write_csv': True,
        'write_plots': False,
        'plot_formats': [],
    }
)

print('Guided-style request:')
print(json.dumps(guided_request, indent=2))

configs = build_configs(ROOT, guided_request, PYTHON_BIN)
for config in configs:
    config['output']['write_plots'] = False
    config['output']['plot_formats'] = []
    config['asymmetric']['beta_equilibrium'] = False
    config['asymmetric']['proton_fraction_values'] = DENSE_Y_VALUES
    config['quarkyonic']['lambda_momentum_mev'] = 200.0
    config['quarkyonic']['n_min_ratio'] = 0.05
    config['quarkyonic']['n_max_ratio'] = 5.0
    config['quarkyonic']['n_points'] = 120
    config['quarkyonic']['fq_scan_points'] = 121
    config['quarkyonic']['shell_integral_points'] = 400
    config['quarkyonic']['quark_integral_points'] = 400
    config['quarkyonic']['smoothing_window'] = 9
    config['quarkyonic']['smoothing_degree'] = 3

print(f'Generated {len(configs)} config(s):')
for config in configs:
    print(f"- {config['run_name']} | branch={config['asymmetric']['branch_mode']} | y-count={len(config['asymmetric']['proton_fraction_values'])}")

config_paths = save_configs(ROOT, configs)
for path in config_paths:
    print(f'Saved config: {path.relative_to(ROOT)}')

In [ ]:
FORCE_RERUN = False

def summary_path_for_config(config):
    return ROOT / config['output']['directory'] / f"{config['run_name']}_summary.json"

summary_paths = [summary_path_for_config(config) for config in configs]
need_run = FORCE_RERUN or not all(path.exists() for path in summary_paths)

if need_run:
    run_results = run_configs(ROOT, config_paths, PYTHON_BIN)
else:
    run_results = []
    for path in summary_paths:
        run_results.append(
            {
                'command': None,
                'stdout': '',
                'stderr': '',
                'summary': json.loads(path.read_text()),
                'summary_path': path,
            }
        )

for result in run_results:
    print('Command used:' if result['command'] else 'Loaded existing summary:')
    print(result['command'] or result['summary_path'].relative_to(ROOT))
    print()
    print(summarize_run(result['summary']))
    print()

In [ ]:
def resolve_output_path(path_text):
    path = Path(path_text)
    return path if path.is_absolute() else ROOT / path


def load_profile_csv(path):
    with path.open('r', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    n_over_n0 = np.array([float(row['n_over_n0']) for row in rows], dtype=float)
    n_b = np.array([float(row['n_b']) for row in rows], dtype=float)
    energy_density = np.array([float(row['energy_density']) for row in rows], dtype=float)
    vs2 = np.array([
        float(row['vs2']) if row['vs2'] not in {'', 'nan', 'None'} else np.nan
        for row in rows
    ], dtype=float)
    order = np.argsort(n_over_n0)
    return {
        'path': path,
        'y': float(rows[0]['y']),
        'n_over_n0': n_over_n0[order],
        'n_b': n_b[order],
        'energy_density': energy_density[order],
        'vs2': vs2[order],
    }


def load_branch_bundle(run_result):
    summary = run_result['summary']
    outputs = summary['outputs']
    branch_mode = summary['asymmetric_fit']['branch_mode']
    profiles = {}
    for key, path_text in sorted(outputs.items()):
        if key.startswith('asymmetric_y_') and key.endswith('_csv'):
            profile = load_profile_csv(resolve_output_path(path_text))
            profiles[round(profile['y'], 3)] = profile
    return {
        'branch_mode': branch_mode,
        'summary': summary,
        'fit': summary['asymmetric_fit'],
        'profiles': profiles,
    }


branch_results = {
    bundle['branch_mode']: bundle
    for bundle in (load_branch_bundle(result) for result in run_results)
}

for branch_mode in ['equal_b', 'target_l']:
    bundle = branch_results[branch_mode]
    y_labels = ', '.join(f'{value:.3f}' for value in sorted(bundle['profiles']))
    print(f"{branch_mode}: {len(bundle['profiles'])} fixed-y curves")
    print(f"  available y values: {y_labels}")
    print(f"  output directory: {resolve_output_path(bundle['summary']['outputs']['summary_json']).parent.relative_to(ROOT)}")

In [ ]:
plt.rcParams.update(
    {
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 12,
        'legend.fontsize': 9,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'axes.spines.top': True,
        'axes.spines.right': True,
    }
)

CMAP = plt.get_cmap('RdYlGn')
NORM = Normalize(vmin=0.0, vmax=0.5)
BRANCH_TEXT = {
    'equal_b': r'$b_{pn}=b_n$',
    'target_l': r'$b_{pn} \neq b_n$',
}


def color_for_y(y_value):
    return CMAP(NORM(float(y_value)))


def energy_per_baryon_gev(profile):
    return profile['energy_density'] / profile['n_b'] / 1000.0 - DEFAULT_PHYSICAL_CONSTANTS.m_nucleon / 1000.0


def peak_point(profile):
    x_values = np.asarray(profile['n_over_n0'], dtype=float)
    y_values = np.asarray(profile['vs2'], dtype=float)
    finite_mask = np.isfinite(x_values) & np.isfinite(y_values)
    if np.count_nonzero(finite_mask) == 0:
        return np.nan, np.nan

    finite_indices = np.where(finite_mask)[0]
    peak_local = int(np.nanargmax(y_values[finite_mask]))
    peak_index = int(finite_indices[peak_local])
    x_peak = float(x_values[peak_index])
    y_peak = float(y_values[peak_index])

    # Refine the peak with a local quadratic fit so the inset does not
    # look artificially stepped by the discrete density grid.
    if 0 < peak_index < len(x_values) - 1:
        window = slice(peak_index - 1, peak_index + 2)
        x_window = x_values[window]
        y_window = y_values[window]
        if np.all(np.isfinite(x_window)) and np.all(np.isfinite(y_window)):
            a_coef, b_coef, c_coef = np.polyfit(x_window, y_window, 2)
            if abs(a_coef) > 1.0e-12 and a_coef < 0.0:
                x_vertex = -b_coef / (2.0 * a_coef)
                if float(x_window[0]) <= x_vertex <= float(x_window[-1]):
                    y_vertex = a_coef * x_vertex * x_vertex + b_coef * x_vertex + c_coef
                    if np.isfinite(y_vertex):
                        x_peak = float(x_vertex)
                        y_peak = float(y_vertex)

    return x_peak, y_peak


def add_peak_inset(parent_ax, profiles, rect, colorbar_rect):
    inset = parent_ax.inset_axes(rect)
    inset.set_facecolor('white')
    y_values = np.array(sorted(profiles), dtype=float)
    peak_n = np.array([peak_point(profiles[round(value, 3)])[0] for value in y_values], dtype=float)
    peak_vs2 = np.array([peak_point(profiles[round(value, 3)])[1] for value in y_values], dtype=float)

    if len(y_values) >= 2:
        points = np.column_stack([peak_n, peak_vs2]).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        line = LineCollection(segments, cmap=CMAP, norm=NORM, linewidths=2.0)
        line.set_array(0.5 * (y_values[:-1] + y_values[1:]))
        inset.add_collection(line)
    inset.scatter(peak_n, peak_vs2, c=y_values, cmap=CMAP, norm=NORM, s=15, edgecolors='none', zorder=3)

    finite_mask = np.isfinite(peak_n) & np.isfinite(peak_vs2)
    x_min = float(np.min(peak_n[finite_mask]))
    x_max = float(np.max(peak_n[finite_mask]))
    y_min = float(np.min(peak_vs2[finite_mask]))
    y_max = float(np.max(peak_vs2[finite_mask]))
    x_pad = 0.06 if x_max > x_min else 0.12
    y_pad = 0.02 if y_max > y_min else 0.04
    inset.set_xlim(x_min - x_pad, x_max + x_pad)
    inset.set_ylim(y_min - y_pad, y_max + y_pad)
    inset.tick_params(axis='both', labelsize=8, pad=1)
    inset.grid(alpha=0.18)
    for spine in inset.spines.values():
        spine.set_linewidth(0.8)

    cax = parent_ax.inset_axes(colorbar_rect)
    scalar = ScalarMappable(norm=NORM, cmap=CMAP)
    colorbar = parent_ax.figure.colorbar(scalar, cax=cax, orientation='horizontal', ticks=np.linspace(0.0, 0.5, 6))
    colorbar.ax.invert_xaxis()
    colorbar.ax.tick_params(labelsize=7, pad=1)
    colorbar.ax.set_title('y', fontsize=8, pad=2)
    colorbar.outline.set_linewidth(0.6)
    return inset


fig, axes = plt.subplots(2, 2, figsize=(11.4, 8.0), sharex=True)
fig.subplots_adjust(left=0.10, right=0.98, bottom=0.10, top=0.96, wspace=0.18, hspace=0.18)

panel_setup = [
    ('equal_b', axes[0, 0], axes[0, 1], '(a)', '(b)'),
    ('target_l', axes[1, 0], axes[1, 1], '(c)', '(d)'),
]

for branch_mode, energy_ax, vs_ax, energy_tag, vs_tag in panel_setup:
    profiles = branch_results[branch_mode]['profiles']
    for y_value in MAIN_Y_VALUES:
        profile = profiles[round(y_value, 3)]
        energy_ax.plot(
            profile['n_over_n0'],
            energy_per_baryon_gev(profile),
            color=color_for_y(y_value),
            lw=1.9,
            label=f'{y_value:.1f}',
        )
        vs_ax.plot(
            profile['n_over_n0'],
            profile['vs2'],
            color=color_for_y(y_value),
            lw=1.9,
        )

    energy_ax.set_xlim(0.0, 5.0)
    energy_ax.set_ylim(-0.02, 0.50)
    energy_ax.grid(alpha=0.20)
    energy_ax.text(0.06, 0.90, 'y =', transform=energy_ax.transAxes, fontsize=9)
    energy_ax.text(0.33, 0.60, 'trivial model\n' + BRANCH_TEXT[branch_mode], transform=energy_ax.transAxes, fontsize=10)
    energy_ax.text(0.95, 0.06, energy_tag, transform=energy_ax.transAxes, ha='right', fontsize=10)

    vs_ax.set_xlim(0.0, 5.0)
    vs_ax.set_ylim(-0.02, 1.00)
    vs_ax.axhline(1.0 / 3.0, color='gray', ls=(0, (4, 4)), lw=0.9)
    vs_ax.grid(alpha=0.20)
    vs_ax.text(0.08, 0.84, r'$\Lambda = 0.2$ GeV', transform=vs_ax.transAxes, fontsize=10)
    vs_ax.text(0.95, 0.06, vs_tag, transform=vs_ax.transAxes, ha='right', fontsize=10)

    add_peak_inset(vs_ax, profiles, rect=[0.58, 0.58, 0.34, 0.30], colorbar_rect=[0.55, 0.49, 0.39, 0.05])

axes[0, 0].legend(title='y', loc='upper left', frameon=False)

for ax in axes[1, :]:
    ax.set_xlabel(r'$\rho_B\ [\rho_0]$')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\epsilon / \rho_B - m_N$ [GeV]')
for ax in axes[:, 1]:
    ax.set_ylabel(r'$v_s^2$')

figure_png = NOTEBOOK_OUTPUT_DIR / 'guided_tvm_fixed_y_replication.png'
figure_pdf = NOTEBOOK_OUTPUT_DIR / 'guided_tvm_fixed_y_replication.pdf'
fig.savefig(figure_png, dpi=240)
fig.savefig(figure_pdf)
plt.show()

print(f'Saved: {figure_png.relative_to(ROOT)}')
print(f'Saved: {figure_pdf.relative_to(ROOT)}')

In [ ]:
peak_rows = []
for branch_mode in ['equal_b', 'target_l']:
    for y_value in sorted(branch_results[branch_mode]['profiles']):
        profile = branch_results[branch_mode]['profiles'][y_value]
        peak_density, peak_vs2 = peak_point(profile)
        peak_rows.append(
            {
                'branch_mode': branch_mode,
                'y': round(float(y_value), 3),
                'peak_n_over_n0': round(float(peak_density), 6),
                'peak_vs2': round(float(peak_vs2), 6),
            }
        )

peak_csv = NOTEBOOK_OUTPUT_DIR / 'guided_tvm_fixed_y_replication_peaks.csv'
with peak_csv.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(peak_rows[0].keys()))
    writer.writeheader()
    writer.writerows(peak_rows)

try:
    import pandas as pd
    display(pd.DataFrame(peak_rows))
except Exception:
    from pprint import pprint
    pprint(peak_rows)

print(f'Saved: {peak_csv.relative_to(ROOT)}')